# 4-2 · 한국어 감성 점수 계산과 NSMC 평가 (5단계)

> **목표:** 직접 확장한 KNU 감성사전(`SentiWord_info_extended_v001.json`)을 이용해 리뷰의 감성 점수를 계산하고, **NSMC 정답 레이블과 비교**한다.

## 이번 노트북의 흐름

1. **사전 검증:** KNU 원본 + 확장본, 신규 감성어 및 품사 기록 확인
2. **분석 파이프라인:** Okt 형태소 분석 → 사전 매칭 → 부정어·강조어 → 점수 합산
3. **작은 문장 실험:** 각 감성 표현의 점수 변화와 매칭 근거 직접 확인
4. **NSMC 테스트 평가:** 원본/확장판 × 규칙 적용 전/후 **4가지 조건을 같은 리뷰에서 비교**
5. **오류 분석:** 오분류, 점수 0, 신규 감성어 기여 사례 살펴보기
6. **결과 내보내기:** `data/outputs/sentiment_evaluation/`에 재현 가능한 CSV/JSON 저장

## 학습 범위 및 중요한 한계

- 사전 기반 **규칙형 NLP**다. 머신러닝 모델을 새로 학습하지 않는다.
- 사전에 있는 모든 문자열을 무조건 감성어로 간주하지 않고, Okt의 **형태소+품사**로 매칭한다.
- 정규화된 여러 형태소 표현은 가능한 범위에서 **긴 패턴을 우선** 매칭한다.
- KNU 원본은 수정하지 않는다. 새로 만든 `v001`을 **읽기 전용**으로 사용한다.
- **3~4단계 후보 선정에 NSMC train을 사용했으므로, 이번 최종 평가는 NSMC test로 수행한다.**
- 단어 빈도·긍정/부정 편중 통계는 감성 점수의 '정답'이 아니며, 사전 등록 점수도 오류가 있을 수 있다.

## 0. 파일 배치와 실행 환경

- 노트북 위치: `4-2/notebooks/05_korean_sentiment_scoring_evaluation.ipynb`
- 점수 계산 모듈: `4-2/src/korean_sentiment.py` (**이 파일도 같이 배치해야 한다**)
- 평가 데이터: `4-2/data/datasets/nsmc/ratings_test.txt`
- 사전 원본: `4-2/data/lexicons/ko/SentiWord_info.json`
- 확장 사전: `4-2/data/lexicons/ko/derived/SentiWord_info_extended_v001.json`
- 신규 단어의 품사·검토 기록: `4-2/data/lexicons/ko/derived/custom_entries_v001.csv`
- 실행 커널: `4-2/.venv/bin/python` (KoNLPy, pandas 포함)

**주의:** 별도 `pip install`이나 데이터 다운로드는 노트북에서 수행하지 않는다. 설치가 되어 있지 않다면 이전 환경 설정 스크립트를 실행한다.

In [9]:
from pathlib import Path
import hashlib
import json
import os
import sys
from collections import Counter

import pandas as pd
from IPython.display import display

# VS Code에서 루트를 기준으로 실행해도, notebooks/를 기준으로 실행해도 경로가 같도록 처리한다.
ROOT = next((p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("4-2/ 또는 4-2/notebooks/에서 노트북을 실행하세요.")
DATA = ROOT / "data"
MODULE_DIR = ROOT / "src"
print(f"MODULE_DIR: {MODULE_DIR}")
if not (MODULE_DIR / "korean_sentiment.py").exists():
    raise FileNotFoundError("src/korean_sentiment.py를 프로젝트에 먼저 복사하세요.")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# VS Code에서 커널을 열면 터미널의 JAVA_HOME을 상속하지 않을 수 있다.
# Okt가 시작되기 전에 환경 설정 스크립트가 선택한 JDK를 커널에서도 찾는다.
if not os.environ.get("JAVA_HOME"):
    manifest_path = DATA / "setup_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
    jdk_dir = DATA / f"jdk-{manifest.get('java_major', 25)}"
    java_candidates = [jdk_dir / "Contents" / "Home", jdk_dir]
    if manifest.get("java_home"):
        java_candidates.insert(0, Path(manifest["java_home"]))
    for java_home in java_candidates:
        if (java_home / "bin" / "java").is_file():
            os.environ["JAVA_HOME"] = str(java_home)
            break

print("프로젝트 루트:", ROOT)
print("현재 커널:", sys.executable)
print("Python 가상환경:", sys.prefix)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(설정 없음 — KoNLPy 동작 여부로 확인)"))
print("pandas:", pd.__version__)

MODULE_DIR: /Users/jhj91_093395/Documents/temp_push/src
프로젝트 루트: /Users/jhj91_093395/Documents/temp_push
현재 커널: /Users/jhj91_093395/Documents/temp_push/.venv/bin/python
Python 가상환경: /Users/jhj91_093395/Documents/temp_push/.venv
JAVA_HOME: /Users/jhj91_093395/Documents/temp_push/data/jdk-25/Contents/Home
pandas: 2.3.3


## 1. 입력 파일 로드 및 확장 사전 검증

- `v001`이 **원본 KNU 전체를 유지하고** 신규 감성어를 뒤에 추가했는지 검사한다.
- 이번 노트북은 기본적으로 `v001`을 사용한다. 다른 버전을 검사할 때만 `LEXICON_VERSION`을 변경한다.
- `custom_entries_v001.csv`의 `(token, pos)`는 **Okt STEM 토큰의 품사**다. 확장 사전의 JSON(`word`, `word_root`, `polarity`) 자체에는 품사가 없어 **보조 CSV를 함께 쓰는 것이 정확하다**.
- 오류가 나면 사전을 자동 복구하거나 원본을 변경하지 않는다.

In [10]:
from korean_sentiment import (
    OktTokenizer, SentimentAnalyzer, build_or_load_indexes,
    check_extension, load_records,
)

LEXICON_VERSION = "v001"  # 이후 새 확장판을 만든 경우 v002 등으로 변경
KNU_PATH = DATA / "lexicons/ko/SentiWord_info.json"
DERIVED_DIR = DATA / "lexicons/ko/derived"
EXTENDED_PATH = DERIVED_DIR / f"SentiWord_info_extended_{LEXICON_VERSION}.json"
CUSTOM_PATH = DERIVED_DIR / f"custom_entries_{LEXICON_VERSION}.csv"
META_PATH = DERIVED_DIR / f"metadata_{LEXICON_VERSION}.json"
TEST_PATH = DATA / "datasets/nsmc/ratings_test.txt"
OUT_DIR = DATA / "outputs/sentiment_evaluation"
CACHE_DIR = OUT_DIR / "cache"

for path in (KNU_PATH, EXTENDED_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"필수 파일 없음: {path}")
if not CUSTOM_PATH.is_file():
    print("⚠ 신규 단어 품사 CSV를 찾지 못했습니다. 확장 JSON만으로 분석하되 품사 매칭의 신뢰도가 낮아질 수 있습니다.")

knu_records = load_records(KNU_PATH)
extended_records = load_records(EXTENDED_PATH)
added_records = check_extension(knu_records, extended_records)
print(f"KNU 원본: {len(knu_records):,}개")
print(f"KNU 확장: {len(extended_records):,}개")
print(f"추가 감성어: {len(added_records)}개")
print("확장판 원본 보존 검증: 통과")

if META_PATH.exists():
    manifest = json.loads(META_PATH.read_text(encoding="utf-8"))
    actual_sha = hashlib.sha256(KNU_PATH.read_bytes()).hexdigest()
    if manifest.get("original_sha256") and manifest["original_sha256"] != actual_sha:
        raise ValueError("메타데이터의 원본 KNU SHA-256이 현재 파일과 다릅니다.")
    print("사전 메타데이터:", META_PATH.name)
else:
    print("메타데이터: 없음 (확장판 내용 일치 검사만 수행)")

KNU 원본: 14,843개
KNU 확장: 14,894개
추가 감성어: 51개
확장판 원본 보존 검증: 통과
사전 메타데이터: metadata_v001.json


In [11]:
# 4단계에서 사람이 승인한 신규 항목을 실제로 살펴본다.
new_words = pd.DataFrame(added_records)
new_words["polarity"] = pd.to_numeric(new_words["polarity"], errors="raise")
print("신규 사전 점수 분포:")
display(new_words["polarity"].value_counts().sort_index().rename("count").to_frame())

if CUSTOM_PATH.is_file():
    custom_review = pd.read_csv(CUSTOM_PATH, encoding="utf-8-sig", dtype="string", keep_default_na=False)
    print("사용자가 승인한 단어의 품사/근거:")
    cols = [c for c in ("token", "pos", "final_score", "reason") if c in custom_review.columns]
    display(custom_review[cols].head(15))
else:
    display(new_words[["word", "polarity"]].head(15))

신규 사전 점수 분포:


,count
polarity,
-2,12
-1,18
1,7
2,14


사용자가 승인한 단어의 품사/근거:


,token,pos,final_score,reason
0,쓰레기,Noun,-2,AI 초안 검토: 영화·연기 등 작품 품질을 심하게 비난하는 은유 표현이다.
1,별로,Noun,-1,AI 초안 검토: “별로”는 평가적 부정 신호지만 “시리즈별로” 등 다른 용법도 있...
2,명작,Noun,2,AI 초안 검토: 영화·작품의 품질을 높이 평가하는 명시적 긍정 명칭이다.
3,뻔하다,Adjective,-1,AI 초안 검토: 전개가 예측 가능하고 새롭지 않다는 평가 용법에서 약한 부정이다....
4,매력,Noun,1,AI 초안 검토: 배우·캐릭터·작품의 호감을 나타내는 경우가 많지만 “매력이 없다”...
5,추천,Noun,1,AI 초안 검토: 관람을 권하는 경우 호평 신호. “비추천/추천하지 않다”는 부정 ...
6,여운,Noun,1,AI 초안 검토: 영화 리뷰에서는 감동의 잔향이라는 긍정 용법이 많으나 여운 자체는...
7,대박,Noun,2,AI 초안 검토: 대체로 놀라움·칭찬의 긍정 감탄이지만 조롱·반어도 가능하다.
8,막장,Noun,-1,AI 초안 검토: 자극적이고 억지스러운 전개를 비판하는 영화·드라마 용어. 일부 장...
9,수작,Noun,2,AI 초안 검토: 영화·작품의 높은 완성도를 평가하는 긍정 명칭이다.


## 2. 형태소 분석기와 감성 사전의 연결

구현은 별도 `src/korean_sentiment.py`에 있다. 여기서는 **사용 방법과 동작 근거**를 살펴본다.

- **형태소 분석:** `Okt.pos(text, norm=True, stem=True)`
  - 예: `재미있었어요`의 활용형을 표제어에 가까운 형태로 처리한다.
- **사전 색인:** 사전 표현도 동일한 Okt 설정으로 분석해 `(형태소, 품사)` 시퀀스에 등록한다.
  - KNU 단일 단어뿐 아니라 여러 형태소로 이루어진 표현도 검색한다.
  - 중복/충돌 시 원본 내에서는 먼저 등장한 패턴을, 확장판에서는 신규 승인 패턴을 우선한다.
  - 새로운 51개 표현은 가능하면 4단계 `custom_entries`의 **원래 품사**를 사용한다.
- **사전 매칭:** 겹치는 표현은 **긴 표현부터** 선택해 중복 합산을 방지한다.
- **성능 고려:** KNU 약 1.5만 항목의 초기 형태소 분석이 오래 걸릴 수 있어, 색인 결과를 `data/outputs/.../cache/`에 JSON으로 캐시한다.

> 매칭은 문맥을 이해하는 것이 아니야. 특히 띄어쓰기 변화, 비유, 반어법, 부정 범위는 별도 처리 없이는 틀릴 수 있다.

In [12]:
# Java/JVM 초기화는 이 셀에서 처음 수행한다.
tokenizer = OktTokenizer()
texts = [
    "영화를 봤다.",
    "영화가 정말 재미있었어요.",
    "영화가 좋지 않았어요.",
    "나쁘지 않은 작품이네요.",
]
for text in texts:
    tokens = tokenizer.tokenize(text)
    print("문장:", text)
    print("형태소·품사:", [(t.form, t.pos) for t in tokens])
    print()

문장: 영화를 봤다.
형태소·품사: [('영화', 'Noun'), ('를', 'Josa'), ('보다', 'Verb'), ('.', 'Punctuation')]

문장: 영화가 정말 재미있었어요.
형태소·품사: [('영화', 'Noun'), ('가', 'Josa'), ('정말', 'Noun'), ('재미있다', 'Adjective'), ('.', 'Punctuation')]

문장: 영화가 좋지 않았어요.
형태소·품사: [('영화', 'Noun'), ('가', 'Josa'), ('좋다', 'Adjective'), ('않다', 'Verb'), ('.', 'Punctuation')]

문장: 나쁘지 않은 작품이네요.
형태소·품사: [('나쁘다', 'Adjective'), ('않다', 'Verb'), ('작품', 'Noun'), ('이네', 'Josa'), ('요', 'Noun'), ('.', 'Punctuation')]



In [13]:
# 최초 실행은 KNU 원본 전체를 토큰화하여 색인을 만들기 때문에 시간이 걸릴 수 있다.
# 동일한 사전 파일과 같은 KoNLPy 버전으로 재실행하면 캐시를 이용한다.
base_index, extended_index, index_stats = build_or_load_indexes(
    KNU_PATH, EXTENDED_PATH, CUSTOM_PATH, tokenizer, CACHE_DIR,
)
print("색인 생성/재사용 현황:")
for key in ("original_records", "custom_records", "custom_pos_hints",
            "base_patterns", "extended_patterns", "base_normalization_collisions",
            "extended_normalization_collisions", "cache_reused", "cache_path"):
    print(f"  {key}: {index_stats[key]}")

analyzers = {
    "knu_plain": SentimentAnalyzer(base_index, use_rules=False),
    "knu_rules": SentimentAnalyzer(base_index, use_rules=True),
    "extended_plain": SentimentAnalyzer(extended_index, use_rules=False),
    "extended_rules": SentimentAnalyzer(extended_index, use_rules=True),
}
print("비교 모델:", list(analyzers))

색인 생성/재사용 현황:
  original_records: 14843
  custom_records: 51
  custom_pos_hints: 51
  base_patterns: 11812
  extended_patterns: 11863
  base_normalization_collisions: 2845
  extended_normalization_collisions: 2845
  cache_reused: False
  cache_path: /Users/jhj91_093395/Documents/temp_push/data/outputs/sentiment_evaluation/cache/okt_lexicon_index_17d2e18be9dfa6a0.json
비교 모델: ['knu_plain', 'knu_rules', 'extended_plain', 'extended_rules']


## 3. 점수 계산 규칙 이해하기

기본 점수는 사전의 `polarity`다. 감성 표현별 점수는 다음과 같이 조정한다.

\[
 s_i = p_i \times (-1)^{n_i} \times w_i
 \qquad S_{document} = \sum_i s_i
\]

- `p_i`: KNU 또는 신규 감성어의 기본 점수 (`-2, -1, +1, +2`)
- `n_i`: 감성 표현 근처에서 발견된 부정어 횟수. **홀수면 반전, 짝수면 원래 부호** (단일 이중 부정 예시)
- `w_i`: 가장 가까운 강조어·약화어 가중치. 없으면 `1.0`
- 여러 감성 표현이 등장하면 조정된 점수를 **합산**한다.

### 구현한 최소 규칙

- **앞쪽 부정:** `안 좋다`, `못 하다`의 `안`·`못`
- **뒤쪽 부정:** `좋지 않다`, `좋지 못하다`의 `않다`·`못하다`
- **강조/약화:** `매우`, `정말`, `너무`, `아주`, `조금` 등
- **이중 부정:** 앞·뒤 양쪽의 부정을 각각 1회 세어 짝수 반전 처리
- **절 경계:** 마침표·쉼표 등 문장부호와 몇 가지 연결 표현을 경계로 보되, 정교한 한국어 구문 분석은 아님

**제약**

- 부정어의 적용 범위를 완벽하게 판단하지는 않는다. 이 셀의 예시는 구현 범위를 확인하는 용도다.
- 사전에 `좋지 않다`라는 **완성형 감성 표현**이 직접 등록되어 매칭된 경우, 내부 부정어를 다시 뒤집지 않는다.
- `영화는 재미있는데 결말은 별로다` 같은 문장의 전체 합은 서로 다른 감성이 상쇄될 수 있다.
- 강조어와 부정어 단독으로는 감성 점수를 더하지 않는다.

In [14]:
def inspect_sentence(text: str):
    """한 문장의 사전 매칭과 점수 계산을 상세히 출력한다."""
    tokens = tokenizer.tokenize(text)
    print("문장:", text)
    print("형태소:", " / ".join(f"{t.form}<{t.pos}>" for t in tokens))
    summary_rows = []
    for name, analyzer in analyzers.items():
        result = analyzer.score_tokens(tokens)
        summary_rows.append({
            "configuration": name,
            "score": result["score"],
            "hits": result["hit_count"],
            "custom_hits": result["custom_hit_count"],
        })
    display(pd.DataFrame(summary_rows))
    detail = analyzers["extended_rules"].score_tokens(tokens)
    print("확장 사전 + 규칙 사용 시 표현별 계산:")
    if detail["hits"]:
        display(pd.DataFrame(detail["hits"])[[
            "term", "matched_tokens", "source", "base_score", "negation_count",
            "modifier", "multiplier", "adjusted_score"
        ]])
    else:
        print("  일치하는 감성 표현이 없습니다. 점수 = 0")

for example in [
    "영화가 정말 재미있었어요.",
    "영화가 좋지 않았어요.",
    "영화가 나쁘지 않았어요.",
    "영화가 안 나쁘지 않았어요.",
]:
    inspect_sentence(example)
    print("=" * 60)

문장: 영화가 정말 재미있었어요.
형태소: 영화<Noun> / 가<Josa> / 정말<Noun> / 재미있다<Adjective> / .<Punctuation>


,configuration,score,hits,custom_hits
0,knu_plain,2.0,1,0
1,knu_rules,3.0,1,0
2,extended_plain,2.0,1,0
3,extended_rules,3.0,1,0


확장 사전 + 규칙 사용 시 표현별 계산:


,term,matched_tokens,source,base_score,negation_count,modifier,multiplier,adjusted_score
0,재미있게,재미있다,knu,2,0,정말,1.5,3.0


문장: 영화가 좋지 않았어요.
형태소: 영화<Noun> / 가<Josa> / 좋다<Adjective> / 않다<Verb> / .<Punctuation>


,configuration,score,hits,custom_hits
0,knu_plain,-2.0,1,0
1,knu_rules,-2.0,1,0
2,extended_plain,-2.0,1,0
3,extended_rules,-2.0,1,0


... (6 outputs left out)

문장: 영화가 안 나쁘지 않았어요.
형태소: 영화<Noun> / 가<Josa> / 안<Noun> / 나쁘다<Adjective> / 않다<Verb> / .<Punctuation>


,configuration,score,hits,custom_hits
0,knu_plain,1.0,1,0
1,knu_rules,1.0,1,0
2,extended_plain,1.0,1,0
3,extended_rules,1.0,1,0


확장 사전 + 규칙 사용 시 표현별 계산:


,term,matched_tokens,source,base_score,negation_count,modifier,multiplier,adjusted_score
0,나쁘지 않고,나쁘다 / 않다,knu,1,0,None,1.0,1.0


## 4. NSMC **test** 데이터 준비

- 감성 사전 후보는 NSMC **train**에서 선정했으므로, **test를 평가에 사용**한다.
- 처음에는 빠르게 실험하도록 **긍정 500개 + 부정 500개**를 무작위로 추출한다.
- `TEST_ROWS_PER_CLASS = None`이면 파일의 모든 유효한 리뷰를 평가한다.
- 난수 시드(`RANDOM_STATE`)를 고정해 여러 분석 조건에 **동일한 리뷰**가 들어가게 한다.
- 리뷰를 제거하거나 레이블을 재작성하지 않는다. 다만 빈 문장·잘못된 레이블은 제외한다.
- 테스트 결과를 보고 **임계값이나 사전을 계속 수정하면 테스트 데이터 누수**가 발생하므로, 수정·튜닝은 train에서만 수행해야 한다.

In [15]:
TEST_ROWS_PER_CLASS = 500  # None: NSMC test 전체 / 100: 긍정·부정 각 100건
RANDOM_STATE = 42

raw_test = pd.read_csv(TEST_PATH, sep="\t", encoding="utf-8", dtype={"id": "string", "document": "string"})
required = {"id", "document", "label"}
if not required.issubset(raw_test.columns):
    raise ValueError(f"NSMC에 필요한 열이 없습니다: {sorted(required - set(raw_test.columns))}")

raw_test["label"] = pd.to_numeric(raw_test["label"], errors="coerce")
valid = raw_test.dropna(subset=["document", "label"]).copy()
valid["document"] = valid["document"].astype("string")
valid = valid.loc[valid["document"].str.strip().ne("") & valid["label"].isin([0, 1])].copy()
valid["label"] = valid["label"].astype("int64")

if TEST_ROWS_PER_CLASS is None:
    test_df = valid.copy()
else:
    chunks = []
    for label in (0, 1):
        group = valid.loc[valid["label"] == label]
        chunks.append(group.sample(n=min(TEST_ROWS_PER_CLASS, len(group)), random_state=RANDOM_STATE))
    test_df = pd.concat(chunks).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

test_df = test_df.reset_index(drop=True)
print("NSMC test 전체:", len(raw_test))
print("사용 가능한 리뷰:", len(valid))
print("이번 평가 표본:", len(test_df))
display(test_df["label"].map({0: "부정", 1: "긍정"}).value_counts().rename("count").to_frame())
display(test_df.head(5))

NSMC test 전체: 50000
사용 가능한 리뷰: 49997
이번 평가 표본: 1000


,count
label,
긍정,500
부정,500


,id,document,label
0,1018235,다세포를 위해 희생하거라.,1
1,2464961,굉장이 맘에 드는 영화지만 마지막 너무 아쉽다 나는 해피앤딩을 원한다 사랑..,1
2,6005823,대단해,1
3,10006932,플래시백 장면에서 쓰인 독특한 촬영기교와 스타일리쉬한 카메라 앵글이 퍽 인상적인 작...,1
4,10009759,와 어쩜 이렇게 내가 싫어하는 배우 원투쓰리가 한작품에 모여있을 수 있지??,0


## 5. 동일 리뷰에 4가지 분석 조건 적용

| 설정 | 감성 사전 | 부정·강조 규칙 |
|---|---|---|
| `knu_plain` | 원본 KNU | 미적용 |
| `knu_rules` | 원본 KNU | 적용 |
| `extended_plain` | 확장 KNU | 미적용 |
| `extended_rules` | 확장 KNU | 적용 |

- **한 리뷰는 한 번만 형태소 분석**하고 같은 토큰을 네 조건에 전달한다.
- 결과에는 최종 점수뿐 아니라 **매칭한 표현 수와 신규 감성어 적용 횟수**도 보존한다.
- 점수에 대한 예측 규칙은 이 단계에서 정하지 않고, 다음 단계에서 명시한다.

In [16]:
score_records = []
extended_hit_details = []  # 나중에 오분류 단어/규칙을 분석하기 위한 흔적

for row_number, row in enumerate(test_df.itertuples(index=False), start=1):
    tokens = tokenizer.tokenize(row.document)
    output = {
        "id": str(row.id),
        "text": str(row.document),
        "label": int(row.label),
    }
    for name, analyzer in analyzers.items():
        result = analyzer.score_tokens(tokens)
        output[f"{name}_score"] = result["score"]
        output[f"{name}_hits"] = result["hit_count"]
        if name == "extended_rules":
            output["custom_hit_count"] = result["custom_hit_count"]
            output["custom_terms"] = ", ".join(
                hit["term"] for hit in result["hits"] if hit["source"] == "custom"
            )
            extended_hit_details.append(result["hits"])
    score_records.append(output)
    if row_number % 250 == 0 or row_number == len(test_df):
        print(f"분석 완료: {row_number:,} / {len(test_df):,}")

results = pd.DataFrame(score_records)
assert len(results) == len(test_df) == len(extended_hit_details)
score_cols = [f"{key}_score" for key in analyzers]
print("점수 분포 요약:")
display(results[score_cols].describe().round(3))
display(results[["id", "label", "text", "extended_rules_score", "custom_terms"]].head(10))

분석 완료: 250 / 1,000
분석 완료: 500 / 1,000
분석 완료: 750 / 1,000
분석 완료: 1,000 / 1,000
점수 분포 요약:


,knu_plain_score,knu_rules_score,extended_plain_score,extended_rules_score
count,1000.000,1000.000,1000.000,1000.000
mean,0.404,0.443,0.377,0.417
std,2.627,2.699,2.754,2.828
min,-14.000,-15.000,-14.000,-15.000
25%,-1.000,-1.000,-1.000,-1.000
50%,0.000,0.000,0.000,0.000
75%,2.000,2.000,2.000,2.000
max,14.000,13.000,14.000,13.000


,id,label,text,extended_rules_score,custom_terms
0,1018235,1,다세포를 위해 희생하거라.,0.0,
1,2464961,1,굉장이 맘에 드는 영화지만 마지막 너무 아쉽다 나는 해피앤딩을 원한다 사랑..,0.5,
2,6005823,1,대단해,2.0,
3,10006932,1,플래시백 장면에서 쓰인 독특한 촬영기교와 스타일리쉬한 카메라 앵글이 퍽 인상적인 작...,7.0,
4,10009759,0,와 어쩜 이렇게 내가 싫어하는 배우 원투쓰리가 한작품에 모여있을 수 있지??,-5.0,
5,8621337,1,"""결론은 """"사랑엔 답이없다""""인 것 같군요.""",1.0,
6,58082,1,정말 재밌게 봤던 드라마. 형부와 처제의 애절한 러브스토리♥,2.0,
7,9226632,1,간만에 진짜 제대로된 헐리우드식 블랙코미디,0.3,
8,9870092,1,뼈속 깊이 공감된다... 책임질땐 아무도 없는 저 상황.. 이래서 전임자의 말은 녹...,1.0,
9,2497897,0,이게영화요? 아 제발.. 이런영화 만들려고 쓸데없이 돈쓰지마시길..,-4.0,


## 6. 점수 → 긍정·부정 분류, Accuracy / F1 평가

**NSMC는 긍정(1) / 부정(0)만 있는 이진 레이블 데이터야.** 그러나 사전 분석 점수에는 `0`이 나올 수 있어.

이번 평가 정책을 고정한다.

- `score > 0` → **긍정(1)**
- `score < 0` → **부정(0)**
- `score == 0` → 사전 근거 부족 또는 상쇄로 해석하고 **별도 비율을 반드시 보고한다**.
- 이진 Accuracy/F1 계산에서 `score == 0`은 편의상 **부정(0)**으로 처리한다. 이 정책은 부정적인 데이터 쪽에 편향되므로 결과를 해석할 때 주의한다.

평가할 지표:

- **Accuracy:** 전체 정답률
- **Precision / Recall / F1 (긍정 기준):** 긍정 예측의 정밀도, 긍정 레이블 회수율, 조화평균
- **Macro F1:** 부정 클래스 F1과 긍정 클래스 F1의 평균
- **점수 0 비율:** 규칙으로 감성을 판단하지 못했거나 상쇄된 비율
- **사전 매칭률:** 감성 표현이 하나 이상 매칭된 리뷰 비율

> `0` 처리는 사전에 학습된 최적 임계값이 아니다. 다른 정책과 비교하려면 **NSMC train에서 먼저 결정**하고 test는 최종 확인에만 사용한다.

In [17]:
def binary_metrics(df: pd.DataFrame, config: str) -> dict:
    score = df[f"{config}_score"].astype(float)
    truth = df["label"].astype(int)
    # score == 0 -> 0(부정)이라는 평가용 고정 정책
    predicted = (score > 0).astype(int)
    tp = int(((predicted == 1) & (truth == 1)).sum())
    tn = int(((predicted == 0) & (truth == 0)).sum())
    fp = int(((predicted == 1) & (truth == 0)).sum())
    fn = int(((predicted == 0) & (truth == 1)).sum())
    def calc(precision_num, precision_den, recall_num, recall_den):
        p = precision_num / precision_den if precision_den else 0.0
        r = recall_num / recall_den if recall_den else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f1
    pos_p, pos_r, pos_f1 = calc(tp, tp+fp, tp, tp+fn)
    neg_p, neg_r, neg_f1 = calc(tn, tn+fn, tn, tn+fp)
    return {
        "config": config,
        "n": len(df), "accuracy": (tp+tn)/len(df),
        "positive_precision": pos_p,
        "positive_recall": pos_r,
        "positive_f1": pos_f1,
        "macro_f1": (pos_f1 + neg_f1)/2,
        "score_zero_rate": float(score.eq(0).mean()),
        "lexicon_match_rate": float(df[f"{config}_hits"].gt(0).mean()),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    }

metrics_df = pd.DataFrame([binary_metrics(results, config) for config in analyzers])
show_cols = ["config", "n", "accuracy", "positive_precision", "positive_recall",
             "positive_f1", "macro_f1", "score_zero_rate", "lexicon_match_rate"]
display(metrics_df[show_cols].round(3))

print("최종 비교의 기준: extended_rules vs knu_rules")
base = metrics_df.set_index("config").loc["knu_rules"]
extended = metrics_df.set_index("config").loc["extended_rules"]
for metric in ("accuracy", "positive_f1", "macro_f1", "score_zero_rate"):
    print(f"{metric:>18}: {base[metric]:.3f} -> {extended[metric]:.3f} "
          f"(차이 {extended[metric] - base[metric]:+.3f})")

,config,n,accuracy,positive_precision,positive_recall,positive_f1,macro_f1,score_zero_rate,lexicon_match_rate
0,knu_plain,1000,0.679,0.699,0.628,0.662,0.678,0.244,0.847
1,knu_rules,1000,0.682,0.699,0.640,0.668,0.681,0.232,0.847
2,extended_plain,1000,0.697,0.719,0.646,0.681,0.696,0.227,0.867
3,extended_rules,1000,0.702,0.722,0.656,0.688,0.701,0.221,0.867


최종 비교의 기준: extended_rules vs knu_rules
          accuracy: 0.682 -> 0.702 (차이 +0.020)
       positive_f1: 0.668 -> 0.688 (차이 +0.020)
          macro_f1: 0.681 -> 0.701 (차이 +0.020)
   score_zero_rate: 0.232 -> 0.221 (차이 -0.011)


In [18]:
# 확장판 + 규칙 기준 혼동행렬: 행=정답, 열=예측
truth = results["label"].astype(int)
predicted = (results["extended_rules_score"] > 0).astype(int)
cm = pd.crosstab(
    truth.map({0: "실제 부정", 1: "실제 긍정"}),
    predicted.map({0: "예측 부정", 1: "예측 긍정"}),
).reindex(index=["실제 부정", "실제 긍정"],
          columns=["예측 부정", "예측 긍정"], fill_value=0)
display(cm)

extended_rules_score,예측 부정,예측 긍정
label,,
실제 부정,374,126
실제 긍정,172,328


## 7. 감성어 51개 추가로 긍정·부정 판정이 바뀐 리뷰

같은 리뷰에 같은 부정·강조 규칙을 적용하고 **KNU 원본 사전과 확장 사전**을 비교한다. 아래 두 표는 판정이 바뀐 리뷰 전체를 실제 정답 기준으로 나눈 것이다.

**표 읽는 법** · ‘추가 표현’의 괄호는 부정·강도를 적용한 표현별 점수다. ‘원본 판정’과 ‘확장 판정’의 괄호는 각각의 리뷰 총점이다. 총점이 0보다 크면 긍정, 0 이하면 부정으로 판정한다.

In [19]:
results["knu_pred"] = (results["knu_rules_score"] > 0).astype(int)
results["extended_pred"] = (results["extended_rules_score"] > 0).astype(int)
results["score_delta"] = results["extended_rules_score"] - results["knu_rules_score"]
results["prediction_changed"] = results["knu_pred"].ne(results["extended_pred"])
results["was_correct"] = results["knu_pred"].eq(results["label"])
results["is_correct"] = results["extended_pred"].eq(results["label"])
results["fixed"] = ~results["was_correct"] & results["is_correct"]
results["regressed"] = results["was_correct"] & ~results["is_correct"]

from IPython.display import HTML

print(f"평가 리뷰 {len(results):,}건 · 추가 단어 검출 {results['custom_hit_count'].gt(0).sum():,}건"
      f" · 점수 변화 {results['score_delta'].ne(0).sum():,}건 · 판정 변화 {results['prediction_changed'].sum():,}건")

labels = {0: "부정", 1: "긍정"}
for flag, title in [("fixed", "오답 → 정답"), ("regressed", "정답 → 오답")]:
    cases = results.loc[results[flag]]
    display(HTML(f"<h4>{title} · {len(cases):,}건</h4>"))
    if cases.empty:
        print("해당 리뷰가 없습니다.")
        continue

    rows = []
    for row in cases.itertuples():
        custom_scores = [
            f"{hit['term']} ({hit['adjusted_score']:+g})"
            for hit in extended_hit_details[row.Index] if hit["source"] == "custom"
        ]
        rows.append({
            "리뷰 원문": row.text,
            "추가 표현 (반영 점수)": ", ".join(custom_scores),
            "원본 판정 (총점)": f"{labels[row.knu_pred]} ({row.knu_rules_score:+g})",
            "확장 판정 (총점)": f"{labels[row.extended_pred]} ({row.extended_rules_score:+g})",
            "실제 정답": labels[row.label],
        })
    # 짧은 원문부터 표시한다. 리뷰 텍스트는 HTML 특수문자를 이스케이프한다.
    table = pd.DataFrame(rows).sort_values("리뷰 원문", key=lambda text: text.str.len(), kind="stable")
    display(HTML(table.to_html(index=False, escape=True)))

custom_hits = Counter(hit["term"] for hits in extended_hit_details for hit in hits
                      if hit["source"] == "custom")
frequency = pd.DataFrame(custom_hits.most_common(), columns=["추가 표현", "검출 횟수"])
display(HTML(f"<details><summary>전체 평가 리뷰에서 검출된 추가 표현 · {len(frequency)}종</summary>"
             f"{frequency.to_html(index=False, escape=True)}</details>"))

평가 리뷰 1,000건 · 추가 단어 검출 127건 · 점수 변화 126건 · 판정 변화 34건


오답 → 정답 · 27건

리뷰 원문,추가 표현 (반영 점수),원본 판정 (총점),확장 판정 (총점),실제 정답
졸려서잤을뿐..,졸리다 (-1),긍정 (+1),부정 (+0),부정
정말 쩐다 쩔어;,쩐다 (+1.5),부정 (+0),긍정 (+1.5),긍정
여운이 남는영화...,여운 (+1),부정 (+0),긍정 (+1),긍정
정말 명작이라 생각됩니다.,명작 (+3),부정 (-1),긍정 (+2),긍정
"이건 쓰레기지,,, 인정하자",쓰레기 (-2),긍정 (+1),부정 (-1),부정
페이크다큐 공포물중 단연 수작,수작 (+2),부정 (+0),긍정 (+2),긍정
지루함을 떠나서 집중이 안돼ㅠㅠ,지루함 (-2),긍정 (+1),부정 (-1),부정
와우~ 곱배기이~~~ 이히~~~,와우 (+1),부정 (+0),긍정 (+1),긍정
진짜 죽기전에 봐야할 걸작선~!,걸작 (+2),부정 (-0.3),긍정 (+1.7),긍정
인생영화. 여운이 오래 남을것같다.,여운 (+1),부정 (+0),긍정 (+1),긍정


정답 → 오답 · 7건

리뷰 원문,추가 표현 (반영 점수),원본 판정 (총점),확장 판정 (총점),실제 정답
전개는 뻔하지만 음악이 주는 감동만으로도 모두 보상된다.,뻔하다 (-1),긍정 (+1),부정 (+0),긍정
빵꾸똥꾸갈비대박..그나마간간히전문성우인배한성씨의 나레이션이나와서다행이었음,대박 (+2),부정 (-1),긍정 (+1),부정
휴...보면서 울었어요ㅠㅠ웰메이드 영화! 제목번역이 거지같지만 정말 연기도 최고...ㅠ_ㅠ,거지같다 (-2),긍정 (+1),부정 (-1),긍정
평점 너무 낮네 1편은 그래도 괜찮은 영화였는데8점은 받아야된다고 생각. 나머지 시리즈가 너무 망작이라 그런가,망작 (-3),긍정 (+1),부정 (-2),긍정
나 역시 저평가 되고 있다고 생각한다. 저속하다 생각하나? 지금 개판 현실 그대로인데; 확연히 그 개판현실 드러내주니 많은 생각이 들고 작품성 꽤나 있다 생각한다. 동생이 이게 19세 미만 금지기 아닌 26세 미만의 영화라더라...마음에많이남는다,"개판 (-2), 개판 (-2)",긍정 (+2),부정 (-2),긍정
내가 34년가까이 살면서 SF명작들 좋아하고 에이리언시리즈 광팬인데 솔직히 프메 이거 개졸작 맞다. 리들리스콧이니까 무조건 명작이라고 우기는 애들이 이상한거다. 원숭이도 나무에서 떨어질때가 있는 경우가 이거지. 편집된 영상 봐도 각본이 엉망이더만.,"명작 (+2), 졸작 (-2), 명작 (+2)",부정 (-1),긍정 (+1),부정
"이제보니 리들리 스코트의,엑소더스-<신들과왕들>은 걸작이었어...셀위에다 도대체 뭔 짓거리를 해논거임?응?일본인이 시도한 붓다 애니는,여기에 비하면,디즈니나 픽사 수준일세...그랴~한국판 북두의권/쿵푸보이 친미/들장미소녀 캔디 실사 영화를 연상 시킴",걸작 (+2),부정 (-1),긍정 (+1),부정


추가 표현,검출 횟수
쓰레기,20
별로,14
명작,14
뻔하다,9
추천,8
수작,8
대박,6
여운,5
진부하다,4
매력,4


## 8. 실패 사례를 실제로 읽고 해석하기

- 정확도만 보고 끝내지 말고 **오분류 사례 10건 이상**을 직접 분석해야 한다.
- 가장 먼저 확인할 것은 감성 사전의 매칭 실패인지, 부정·강조 규칙의 실패인지, 문맥 자체가 어려운 사례인지다.
- `custom_hit_count=0`이면서 틀린 리뷰는 **신규 사전 51개가 적용되지 않은 경우**일 수 있다.
- 오류 사유는 기계가 자동으로 확정하지 않는다. 아래 예문을 보고 **사용자가 직접 원인**을 기록한다.

In [20]:
# 데이터프레임 행 번호로 문장 하나를 상세 조사한다.
def inspect_test_row(row_idx: int):
    item = results.iloc[row_idx]
    print("행 번호:", row_idx, "/ id:", item["id"])
    print("문장:", item["text"])
    print("정답:", "긍정" if item["label"] else "부정")
    print("원본+규칙:", item["knu_rules_score"])
    print("확장+규칙:", item["extended_rules_score"])
    print("확장판 예측:", "긍정" if item["extended_pred"] else "부정")
    hits = extended_hit_details[row_idx]
    if hits:
        display(pd.DataFrame(hits)[[
            "term", "source", "base_score", "negation_count",
            "modifier", "multiplier", "adjusted_score"
        ]])
    else:
        print("사전 매칭 항목이 없습니다.")

wrong_idx = results.index[~results["is_correct"]].tolist()
zero_idx = results.index[results["extended_rules_score"].eq(0)].tolist()
print("오분류:", len(wrong_idx), "개 / 점수 0:", len(zero_idx), "개")
if wrong_idx:
    inspect_test_row(wrong_idx[0])
else:
    print("이 표본에서는 오분류가 없습니다.")

오분류: 298 개 / 점수 0: 221 개
행 번호: 0 / id: 1018235
문장: 다세포를 위해 희생하거라.
정답: 긍정
원본+규칙: 0.0
확장+규칙: 0.0
확장판 예측: 부정
사전 매칭 항목이 없습니다.


In [21]:
# 수작업으로 살펴볼 리뷰를 추출한다. 아래 DataFrame은 원인 기록 전 상태이다.
wrong_review = results.loc[~results["is_correct"]].head(30).copy()
wrong_review["error_type"] = ""  # 예: 사전 미등록 / 부정 범위 / 중의성 / 반어 / 혼합 감성
wrong_review["analysis_note"] = ""  # 실제 원문과 매칭된 표현을 보고 직접 작성
columns = ["id", "label", "text", "extended_rules_score", "custom_terms",
           "error_type", "analysis_note"]
print("검토 목록 (상위 30건):")
display(wrong_review[columns].head(15))

# score=0이 많은 경우 실제 평가 지표를 왜곡할 수 있으므로 함께 확인한다.
print("점수 0인 리뷰 예시:")
display(results.loc[results["extended_rules_score"].eq(0),
                    ["label", "text", "extended_rules_hits", "custom_terms"]].head(10))

검토 목록 (상위 30건):


,id,label,text,extended_rules_score,custom_terms,error_type,analysis_note
0,1018235,1,다세포를 위해 희생하거라.,0.0,,,
10,3090441,1,생각만 해도 눈물이 주룩.. 어떻게 이렇게 탄탄한 구성의 드라마를 만들 수 있는지..,0.0,,,
15,1209054,1,너무 끔찍해요...슬퍼요..현실이 저렇다니..,-5.0,,,
16,5887111,0,빵꾸똥꾸갈비대박..그나마간간히전문성우인배한성씨의 나레이션이나와서다행이었음,1.0,대박,,
25,8723117,0,오늘 조조로 봤는데... 영화보다 뛰쳐나갈번함...ㅡㅡ,1.0,,,
27,2115748,1,"한국 사람들, 조금만 겸손하자. 우리도 어려운 시절이 있었다..",-1.5,,,
28,5316290,1,남자로서.. 무...무섭다... ㅠㅠ,-3.0,,,
30,501947,0,국군 공수부대가 뛰어내려도 더 괜찮을 듯...,1.0,,,
36,324585,1,슬픈사랑의 노래... 분위기가 인상적이었던 영화...,0.0,,,
39,5524080,0,재미 없엉.,1.0,,,


점수 0인 리뷰 예시:


,label,text,extended_rules_hits,custom_terms
0,1,다세포를 위해 희생하거라.,0,
10,1,생각만 해도 눈물이 주룩.. 어떻게 이렇게 탄탄한 구성의 드라마를 만들 수 있는지..,2,
11,0,연기가 쩝. 책을읽어라.,0,
33,0,테켄 보다 여자들이 많이 못하고 킹오파 보다는 훨씬 우수. 3개 안봤으면 테켄만보슈,4,
36,1,슬픈사랑의 노래... 분위기가 인상적이었던 영화...,2,
43,1,연기력 하나만으로도 먹어준다.,0,
48,0,영화관에서 보지않은게 천만다행,0,
52,0,역시 서세원틱한 영화..,0,
57,0,평점에 낚여서 본영화. 전 별로였음 복수는 복수 다워야 하는데 이건뭐 1시간짜리로 ...,6,별로
58,0,다이하드5 라는데 정작 다이하드는 안보이고 영웅 존 맥클레인도 없다. 엄청난 제작비...,5,


## 9. 결과 CSV와 실행 메타데이터 저장

`data/outputs/sentiment_evaluation/` 내부에 결과를 저장한다.

- `review_scores_vNNN.csv`: 리뷰별 점수와 예측·신규 단어 영향
- `metrics_vNNN.csv`: 네 조건의 정확도, Precision/Recall/F1, 0점 비율
- `errors_to_review_vNNN.csv`: 오류 유형과 근거를 직접 기록할 템플릿
- `run_metadata_vNNN.json`: 입력 파일 해시, 평가 표본 조건, 규칙/분류 정책

**안전 정책:** 기존 파일은 덮어쓰지 않고 `v001`, `v002`, … 로 번호를 올린다. 결과물은 모두 `data/`에 저장한다. 확장 사전 원본은 변경하지 않는다.

In [22]:
EXPORT_RESULTS = True  # 결과 파일을 저장하지 않고 탐색만 하려면 False

def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

if EXPORT_RESULTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    version = 1
    while any((OUT_DIR / f"{name}_v{version:03d}.{extension}").exists()
              for name, extension in (("review_scores", "csv"),
                                      ("metrics", "csv"),
                                      ("errors_to_review", "csv"),
                                      ("run_metadata", "json"))):
        version += 1
    suffix = f"v{version:03d}"
    outputs = {
        "scores": OUT_DIR / f"review_scores_{suffix}.csv",
        "metrics": OUT_DIR / f"metrics_{suffix}.csv",
        "errors": OUT_DIR / f"errors_to_review_{suffix}.csv",
        "metadata": OUT_DIR / f"run_metadata_{suffix}.json",
    }
    # mode="x": 존재하는 파일이 있으면 실패 (덮어쓰기 방지)
    results.to_csv(outputs["scores"], index=False, encoding="utf-8-sig", mode="x")
    metrics_df.to_csv(outputs["metrics"], index=False, encoding="utf-8-sig", mode="x")
    wrong_review[columns].to_csv(outputs["errors"], index=False, encoding="utf-8-sig", mode="x")
    run_metadata = {
        "original_lexicon": str(KNU_PATH.relative_to(ROOT)),
        "original_sha256": file_sha256(KNU_PATH),
        "extended_lexicon": str(EXTENDED_PATH.relative_to(ROOT)),
        "extended_sha256": file_sha256(EXTENDED_PATH),
        "custom_pos_csv": str(CUSTOM_PATH.relative_to(ROOT)) if CUSTOM_PATH.exists() else None,
        "custom_csv_sha256": file_sha256(CUSTOM_PATH) if CUSTOM_PATH.exists() else None,
        "test_dataset": str(TEST_PATH.relative_to(ROOT)),
        "test_sha256": file_sha256(TEST_PATH),
        "test_rows_per_class": TEST_ROWS_PER_CLASS,
        "random_state": RANDOM_STATE,
        "evaluation_n": len(results),
        "scoring": "Okt norm=True stem=True; longest match; sum; optional simple negation/intensifier rules",
        "binary_threshold": 0,
        "zero_score_tie_policy": "predict negative (0); zero rate separately reported",
        "lexicon_index": index_stats,
        "model_configs": list(analyzers.keys()),
    }
    with outputs["metadata"].open("x", encoding="utf-8") as fh:
        json.dump(run_metadata, fh, ensure_ascii=False, indent=2)
    print("저장 완료:")
    for label, path in outputs.items():
        print(f"- {label}: {path.relative_to(ROOT)}")
else:
    print("EXPORT_RESULTS=False: 파일을 생성하지 않았습니다.")

저장 완료:
- scores: data/outputs/sentiment_evaluation/review_scores_v002.csv
- metrics: data/outputs/sentiment_evaluation/metrics_v002.csv
- errors: data/outputs/sentiment_evaluation/errors_to_review_v002.csv
- metadata: data/outputs/sentiment_evaluation/run_metadata_v002.json


## 10. 결과 해석과 다음 단계

- [ ] 신규 51개 단어 중 실제 테스트 리뷰에 등장한 항목을 확인했다.
- [ ] 기본 KNU vs 확장 KNU **동일한 규칙 조건**에서 Accuracy / Macro F1을 비교했다.
- [ ] 부정·강조 규칙 **적용 전/후**를 비교했다.
- [ ] 점수 `0` 비율을 확인하고 이진 분류의 0점 처리 정책을 이해했다.
- [ ] 오분류 리뷰 10건 이상을 조사할 수 있도록 CSV를 추출했다.
- [ ] 확장판이 도움이 된 사례와 오히려 성능을 악화시킨 사례를 구분했다.

### 중요한 고찰

- **감성 사전 항목을 많이 추가하는 것 ≠ 반드시 성능 향상.** 품사 혼동, 중립 단어의 과잉 점수화, 문맥 의존 표현 때문에 성능이 떨어질 수 있다.
- **사전 매칭률과 분류 정확도는 다르다.** 단어를 잘 찾더라도 잘못된 점수나 규칙을 적용하면 오분류할 수 있다.
- KNU와 승인한 단어는 **문장 레이블을 기반으로 학습한 분류 모델이 아니야.** 사람이 만든 기준을 적용하는 규칙 기반 베이스라인이다.
- 테스트 데이터의 오류를 보고 사전·규칙을 고친다면, 다음 성능 측정에는 **독립된 새 테스트 세트**가 필요하다.
- 다음 확장 방향: 부정 범위 개선, KNU 구절 매칭 검증, 점수 정규화, 0점 처리 정책 개선 및 실제 오류 유형별 분석.